# Clase 3 - Práctica: Visión por computador II, arquitecturas avanzadas

En esta práctica aplicas lo que viste en la Clase 3 sobre redes neuronales convolucionales (CNN) avanzadas: las conexiones residuales, que hacen posible entrenar redes muy profundas, y el aprendizaje por transferencia, que reutiliza un modelo preentrenado para resolver una tarea nueva con pocos datos.

## Qué vas a practicar

- Construyes desde cero un bloque residual en PyTorch y confirmas que la conexión de salto funciona como esperas.
- Adaptas una ResNet-18 preentrenada a un problema de clasificación binaria con pocas imágenes, mediante aprendizaje por transferencia.

## Requisito previo

Antes de empezar, confirma que ya viste la Clase 3, "Visión por computador II: arquitecturas avanzadas". Necesitas entender qué es una conexión residual y en qué consiste el aprendizaje por transferencia para completar los dos ejercicios.

Ejecuta este notebook en Google Colab. No instales nada adicional: PyTorch y torchvision vienen preinstalados en el entorno.

## Ejercicio 1 — Construye un bloque residual desde cero

Tu equipo quiere entender, línea a línea, cómo funciona la pieza central de ResNet (Residual Network, "red residual") antes de usarla como caja negra en un proyecto. Te piden implementar la clase `BloqueResidual`, el componente básico que se repite decenas de veces dentro de una red residual.

Un bloque residual aplica dos convoluciones 3x3 con normalización por lotes (batch normalization) y, antes de la activación final, suma la entrada original del bloque a la salida de esas convoluciones. Esa suma es la conexión residual: en vez de aprender la transformación completa H(x), el bloque aprende solo el residuo F(x) = H(x) − x, y reconstruye la salida como y = F(x) + x.

Cuando el bloque cambia el número de canales o la resolución espacial (por ejemplo, con `stride=2`), la entrada x no tiene la misma forma que F(x), así que no puedes sumarlas directamente. Por eso el andamiaje ya incluye un atajo (`self.atajo`) que aplica una convolución 1x1 y normalización por lotes solo cuando hace falta ajustar esa forma.

**Pista:** el método `__init__` ya está completo. Tu tarea es terminar el método `forward`: falta una única línea, la que suma `salida` e `identidad` antes de devolver el resultado con la activación aplicada.

In [ ]:
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Bloque residual básico: dos convoluciones 3x3 y un atajo (shortcut)
    que suma la entrada a la salida antes de la última activación."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        # Si el bloque cambia la resolución o el número de canales, el atajo
        # necesita una proyección 1x1 para poder sumarse a F(x) más adelante
        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)                # esta es la vía directa: F(x) + x
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))

        # TODO: suma la conexión residual antes de la activación final.
        # Combina "salida" e "identidad" y guarda el resultado en "salida"
        salida = ...

        return self.relu(salida)

In [ ]:
# Prueba rápida: comprueba que el bloque produce la forma de salida esperada
entrada_prueba = torch.randn(2, 3, 32, 32)  # lote de 2 imágenes, 3 canales, 32x32 píxeles
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
print(bloque(entrada_prueba).shape)  # forma esperada: torch.Size([2, 16, 16, 16])

<details>
<summary>💡 Ver solución</summary>

```python
import torch
import torch.nn as nn


class BloqueResidual(nn.Module):
    """Bloque residual básico: dos convoluciones 3x3 y un atajo (shortcut)
    que suma la entrada a la salida antes de la última activación."""

    def __init__(self, canales_entrada, canales_salida, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(canales_entrada, canales_salida, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(canales_salida)
        self.conv2 = nn.Conv2d(canales_salida, canales_salida, kernel_size=3,
                                stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(canales_salida)
        self.relu = nn.ReLU(inplace=True)

        self.atajo = nn.Sequential()
        if stride != 1 or canales_entrada != canales_salida:
            self.atajo = nn.Sequential(
                nn.Conv2d(canales_entrada, canales_salida, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(canales_salida),
            )

    def forward(self, x):
        identidad = self.atajo(x)
        salida = self.relu(self.bn1(self.conv1(x)))
        salida = self.bn2(self.conv2(salida))

        # La conexión residual: sumamos la entrada (ajustada por el atajo si
        # hizo falta) a lo que calcularon las dos convoluciones, y solo
        # entonces aplicamos la última activación
        salida = salida + identidad

        return self.relu(salida)


entrada_prueba = torch.randn(2, 3, 32, 32)
bloque = BloqueResidual(canales_entrada=3, canales_salida=16, stride=2)
print(bloque(entrada_prueba).shape)  # torch.Size([2, 16, 16, 16])
```

Si comentas la línea `salida = salida + identidad`, el bloque sigue ejecutándose sin errores, pero deja de ser residual: cada bloque tendría que reconstruir la transformación completa desde cero, en vez de aprender solo la diferencia respecto a dejar pasar la entrada. Con redes de decenas de capas, esa diferencia es la que marca si el entrenamiento converge o se estanca.
</details>

## Ejercicio 2 — Clasificador de dos categorías con aprendizaje por transferencia

Una entidad te pide un clasificador que distinga fotografías de dos categorías de producto, pero solo dispone de un puñado de imágenes etiquetadas por categoría: no alcanza para entrenar una red convolucional desde cero. Aplicas aprendizaje por transferencia: partes de una ResNet-18 ya entrenada sobre ImageNet, conservas todo lo que aprendió sobre bordes, texturas y formas, y entrenas únicamente una capa nueva adaptada a tu problema.

Para esta práctica simulas ese escenario con el conjunto CIFAR-10: te quedas solo con dos de sus diez clases (gato y perro) y con un subconjunto reducido de imágenes por clase, para reproducir la situación de "pocos datos".

El andamiaje de carga de datos y el bucle de entrenamiento ya están completos. Te falta la parte central de la transferencia, en la celda que define el modelo:

1. Congela todos los parámetros de la ResNet-18 preentrenada, para que el entrenamiento no los modifique.
2. Sustituye la última capa (`modelo.fc`) por una capa lineal nueva, adaptada al número de clases de tu problema. Esta capa nueva sí se entrena: al crearla desde cero, sus parámetros nacen con `requires_grad = True`.

**Pista:** para congelar un parámetro en PyTorch, pon su atributo `.requires_grad` en `False`. Recorre `modelo.parameters()` con un bucle. La capa `modelo.fc` original es un `nn.Linear` que recibe `modelo.fc.in_features` características de entrada; usa ese mismo número al crear la capa de reemplazo.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, Dataset

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dos categorías de CIFAR-10: 3 = gato, 5 = perro
CLASES_ELEGIDAS = [3, 5]
NUM_CLASES = len(CLASES_ELEGIDAS)
IMAGENES_POR_CLASE = 200  # subconjunto pequeño, para simular un escenario de pocos datos

preprocesar = transforms.Compose([
    transforms.Resize(224),  # ResNet-18 espera imágenes de 224x224 píxeles
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # medias de ImageNet
                          std=[0.229, 0.224, 0.225]),
])

dataset_completo = torchvision.datasets.CIFAR10(
    root="./datos", train=True, download=True, transform=preprocesar
)

# Nos quedamos solo con las imágenes de las dos clases elegidas
etiquetas = torch.tensor(dataset_completo.targets)
indices_filtrados = []
for clase in CLASES_ELEGIDAS:
    indices_clase = (etiquetas == clase).nonzero(as_tuple=True)[0].tolist()
    indices_filtrados.extend(indices_clase[:IMAGENES_POR_CLASE])

subconjunto = Subset(dataset_completo, indices_filtrados)


class DatasetDosClases(Dataset):
    """Envuelve el subconjunto y remapea las etiquetas originales (3, 5)
    a (0, 1), que es lo que espera una capa lineal de dos salidas."""

    def __init__(self, subconjunto, clases_elegidas):
        self.subconjunto = subconjunto
        self.clases_elegidas = clases_elegidas

    def __len__(self):
        return len(self.subconjunto)

    def __getitem__(self, idx):
        imagen, etiqueta_original = self.subconjunto[idx]
        etiqueta_nueva = self.clases_elegidas.index(etiqueta_original)
        return imagen, etiqueta_nueva


dataset_entrenamiento = DatasetDosClases(subconjunto, CLASES_ELEGIDAS)
cargador_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=16, shuffle=True)

print(f"Imágenes de entrenamiento: {len(dataset_entrenamiento)}")

In [ ]:
modelo = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

# TODO: congela todos los parámetros de la ResNet-18 preentrenada.
# Recorre modelo.parameters() con un bucle y pon requires_grad = False
# en cada parámetro, para que el entrenamiento no los modifique
...

# La capa modelo.fc original es un nn.Linear que mapea a las 1000 clases de ImageNet.
# TODO: sustitúyela por un nn.Linear nuevo, con modelo.fc.in_features características
# de entrada y NUM_CLASES de salida. Al crearse desde cero, sus parámetros nacen
# con requires_grad = True, así que serán los únicos que se entrenen
modelo.fc = ...

modelo = modelo.to(dispositivo)

In [ ]:
criterio = nn.CrossEntropyLoss()
# Solo optimizamos los parámetros de la capa nueva: el resto quedó congelado
optimizador = optim.Adam(modelo.fc.parameters(), lr=1e-3)

modelo.train()
NUM_EPOCAS = 3
for epoca in range(NUM_EPOCAS):
    perdida_acumulada = 0.0
    for imagenes, etiquetas_lote in cargador_entrenamiento:
        imagenes = imagenes.to(dispositivo)
        etiquetas_lote = etiquetas_lote.to(dispositivo)

        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas_lote)
        perdida.backward()
        optimizador.step()

        perdida_acumulada += perdida.item()

    perdida_media = perdida_acumulada / len(cargador_entrenamiento)
    print(f"Época {epoca + 1}/{NUM_EPOCAS} - pérdida media: {perdida_media:.4f}")

<details>
<summary>💡 Ver solución</summary>

```python
modelo = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

# Congelamos todos los parámetros preentrenados: no queremos que el
# entrenamiento con pocos datos los modifique y arruine lo ya aprendido
for parametro in modelo.parameters():
    parametro.requires_grad = False

# Sustituimos la última capa por una nueva, adaptada a nuestras dos clases.
# Sus parámetros nacen con requires_grad = True, así que serán los únicos
# que el optimizador actualice durante el entrenamiento
modelo.fc = nn.Linear(modelo.fc.in_features, NUM_CLASES)

modelo = modelo.to(dispositivo)
```

Fíjate en el resultado: aunque la ResNet-18 completa tiene millones de parámetros, solo entrenas los de la capa `fc` nueva, unos pocos miles. Por eso el aprendizaje por transferencia funciona con tan pocas imágenes: el modelo no tiene que reaprender a detectar bordes ni texturas, solo a combinar las características que ya extrae para separar tus dos categorías.
</details>

## Para cerrar

Acabas de poner en práctica dos ideas centrales de la visión por computador moderna: la conexión residual, que permite entrenar redes muy profundas, y el aprendizaje por transferencia, que ahorra datos y cómputo al reutilizar lo que otro modelo ya aprendió. Son las mismas piezas que sostienen detectores como Faster R-CNN o RetinaNet y redes de segmentación como U-Net, arquitecturas que verás más adelante en el curso.

Si algo no funcionó a la primera, revisa las formas de los tensores en cada paso: la mayoría de errores en estos ejercicios vienen de una discrepancia entre canales o entre dimensiones.

**Abre el notebook en Google Colab, completa los dos ejercicios sin mirar la solución antes de intentarlo y comparte tus dudas en el foro de la clase.**